In [ ]:
# Installations-required
!pip install biopython
!apt-get -qq install cd-hit
!pip install openpyxl tqdm


In [ ]:
# IMPORT LIBRARIES
import pandas as pd
import os
from Bio.PDB import MMCIFParser
from Bio.SeqUtils import seq1
from tqdm import tqdm

# INPUT SETTINGS
INPUT_FILE = "binding_sites.xlsx"
CIF_FOLDER = "/path/to/cif/files/"
FASTA_FILE = "all_sequences.fasta"
CLUSTERED_FASTA = "clustered.fasta"
CLUSTER_FILE = "clustered.fasta.clstr"
OUTPUT_FILE = "nonredundant_binding_sites.xlsx"

TEST_ROWS = None

# LOAD DATA
df = pd.read_excel(INPUT_FILE)

if TEST_ROWS:
    df = df.head(TEST_ROWS)

parser = MMCIFParser(QUIET=True)

# HELPER FUNCTIONS
def parse_pdb_chain(entry):
    pdb, resid, chain = entry.split("_")
    return pdb, chain

def extract_sequence(pdb_id, chain_id):

    cif_path = os.path.join(CIF_FOLDER, f"{pdb_id}.cif")

    if not os.path.exists(cif_path):
        return ""

    structure = parser.get_structure(pdb_id, cif_path)

    seq = []

    for model in structure:
        for chain in model:

            if chain.id != chain_id:
                continue

            for residue in chain:

                resname = residue.get_resname()

                if resname in ["HOH","HO","O"]:
                    continue

                try:
                    aa = seq1(resname)
                except:
                    continue

                seq.append(aa)

    return "".join(seq)

def build_signature(row):

    residues = []
    chain_map = {}
    next_chain = 0

    for col in row.index:

        if not col.startswith("Residue"):
            continue

        val = row[col]

        if pd.isna(val):
            continue

        val = str(val)

        if val.upper() == "OTHER":
            continue

        parts = val.split("_")

        if len(parts) != 3:
            continue

        resname, resid, chain = parts

        if chain not in chain_map:
            chain_map[chain] = chr(ord('X') + next_chain)
            next_chain += 1

        canon_chain = chain_map[chain]

        residues.append(f"{resname}{resid}_{canon_chain}")

    residues.sort()

    return "-".join(residues)

# EXTRACT SEQUENCES
print("Extracting sequences...")

seq_ids = []
sequences = []
signatures = []

for idx,row in tqdm(df.iterrows(), total=len(df)):

    pdb,chain = parse_pdb_chain(row["PDB_Copper_Chain"])

    seq = extract_sequence(pdb,chain)

    seq_ids.append(f"{pdb}_{chain}")
    sequences.append(seq)
    signatures.append(build_signature(row))

df["sequence"] = sequences
df["signature"] = signatures
df["seq_id"] = seq_ids

# WRITE FASTA
print("Writing FASTA file...")

with open(FASTA_FILE,"w") as f:

    for sid,seq in zip(seq_ids,sequences):

        f.write(f">{sid}\n{seq}\n")

# RUN CD-HIT
print("Running CD-HIT clustering...")

!cd-hit -i {FASTA_FILE} -o {CLUSTERED_FASTA} -c 0.9 -n 5

# PARSE CLUSTER FILE
print("Parsing clusters...")

clusters = []
current_cluster = []

with open(CLUSTER_FILE) as f:

    for line in f:

        if line.startswith(">Cluster"):

            if current_cluster:
                clusters.append(current_cluster)

            current_cluster = []

        else:

            seq_id = line.split(">")[1].split("...")[0]

            current_cluster.append(seq_id)

    if current_cluster:
        clusters.append(current_cluster)

# APPLY BINDING SITE FILTER
print("Applying binding-site rules...")

keep_ids = []

for cluster in clusters:

    seen = []

    for sid in cluster:

        row = df[df.seq_id == sid].iloc[0]

        signature = row["signature"]

        cu_type = row["CU_Type"] if "CU_Type" in df.columns else None

        duplicate = False

        for prev_sid in seen:

            prev_row = df[df.seq_id == prev_sid].iloc[0]

            same_signature = signature == prev_row["signature"]

            if "CU_Type" in df.columns:
                same_type = cu_type == prev_row["CU_Type"]
            else:
                same_type = True

            if same_signature and same_type:
                duplicate = True
                break

        if not duplicate:
            keep_ids.append(sid)
            seen.append(sid)

# BUILD FINAL DATASET
df_final = df[df.seq_id.isin(keep_ids)].reset_index(drop=True)

df_final.to_excel(OUTPUT_FILE,index=False)

print("\nOriginal entries:",len(df))
print("Nonredundant entries:",len(df_final))
print("Saved:",OUTPUT_FILE)

# To handle water involved binding sites
# Load the nonredundant dataset
df = pd.read_excel("nonredundant_binding_sites.xlsx")

# Function to normalize signature ignoring HOH residue numbers
def normalize_signature(sig):

    parts = sig.split("-")
    norm = []

    for p in parts:

        if p.startswith("HOH"):
            chain = p.split("_")[-1]
            norm.append(f"HOH_{chain}")   # ignore water residue number

        else:
            norm.append(p)

    norm.sort()

    return "-".join(norm)


# Create normalized signature column
df["normalized_signature"] = df["signature"].apply(normalize_signature)

# Remove duplicates
if "CU_Type" in df.columns:
    df_clean = df.drop_duplicates(subset=["normalized_signature","CU_Type"])
else:
    df_clean = df.drop_duplicates(subset=["normalized_signature"])

# Save cleaned dataset
df_clean.to_excel("nonredundant_binding_sites_water_fixed.xlsx", index=False)

print("Original entries:", len(df))
print("After water-duplicate removal:", len(df_clean))
print("Saved file: nonredundant_binding_sites_water_fixed.xlsx")

# Read the Excel file
df = pd.read_excel("nonredundant_binding_sites_water_fixed.xlsx")

# Save as CSV
df.to_csv("nonredundant_binding_sites_water_fixed.csv", index=False)